# Bangla Fake News Detection

## Objective
This project is edutational purpose, trained with "BanFakeNews", famous Kaggle Data(50K). 

## Import Necessary libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Import Datasets

In [2]:
auth= pd.read_csv("data/Authentic-48K.csv")
fake= pd.read_csv("data/Fake-1K.csv")

In [3]:
# Check columns
auth.columns.tolist()==fake.columns.tolist()

True

In [4]:
# Concate Both files
news=pd.concat([auth, fake], ignore_index=True)
news

,articleID,domain,date,category,headline,content,label
0,1,jagonews24.com,2018-09-19 17:48:18,Education,"হট্টগোল করায় বাকৃবিতে দুইজন বরখাস্ত, ৬ জনকে শোকজ",গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে ...,1
1,2,jagonews24.com,2018-09-19 17:48:19,National,মালয়েশিয়ায় কর্মী পাঠানোর ব্যবস্থা নেয়ার সুপারিশ,বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রম...,1
2,3,jagonews24.com,2018-09-19 17:48:20,National,প্রেমের প্রস্তাবে রাজি না হওয়ায় স্কুলছাত্রীকে ...,নরসিংদীর মনোহরদীতে প্রেমের প্রস্তাবে রাজি না হ...,1
3,4,jagonews24.com,2018-09-19 17:48:21,Crime,মেডিয়েশনই মামলাজট নিরসনের পথ : বিচারপতি আহমেদ ...,সুপ্রিম কোর্টের হাইকোর্ট বিভাগের বিচারপতি আহমে...,1
4,5,jagonews24.com,2018-09-19 17:48:21,National,টকশোতে বক্তব্য দিতে গিয়ে জাপা নেতার মৃত্যু,মাদারীপুর সদরের উপজেলার লেকেরপাড়ে একটি বেসরকার...,1
...,...,...,...,...,...,...,...
49972,1303,bengalbeats.com,2019-03-31T21:34:42+00:00,Miscellaneous,রাজশাহীতে স্বামীর হাঁচির আওয়াজে ঘুমের মধ্যেই স...,রাজশাহীর সাহেব বাজার মোড়ে রাতে ঘুমের মধ্যে হাস...,0
49973,1304,bengaliviralnews.com,2018-01-16T15:36:12+00:00,Miscellaneous,IAS ইন্টারভিউ এ মেয়েটিকে জিজ্ঞসা করা হল মেয়েদে...,IAS পরীক্ষায় এমন এমন অবাক করে দেওয়া প্রশ্ন এসে...,0
49974,1305,channeldhaka.news,2018-12-03T18:09:08+00:00,Crime,শাওমি মোবাইল বিস্ফোরণে কলেজছাত্রের মৃত্যু!,শাওমি মোবাইল ফোন বিস্ফোরণ হয়ে স্বপ্নীল মজুমদা...,0
49975,1306,earki.com,"১৭:৩৬, জানুয়ারি ০৭, ২০১৯",Miscellaneous,ফেসবুকে যেভাবে নতুন মন্ত্রীদের অভিনন্দন বার্তা...,আপনাকে নতুন মন্ত্রিসভার সদস্যদের অভিনন্দন জানা...,0


In [5]:
news.shape

(49977, 7)

# Class Imbalance Check

In [6]:
news["label"].value_counts()

label
1    48678
0     1299
Name: count, dtype: int64

In [7]:
news["label"].value_counts(normalize=True)*100

label
1    97.400804
0     2.599196
Name: proportion, dtype: float64

## Confirmed — 97.4% authentic vs 2.6% fake, so imbalance-handling isn't optional here; a model that just predicts "authentic" every time would already hit 97.4% accuracy while being useless.

## Next step: before you touch any imbalance-handling technique (SMOTE, class weights, etc.), you need to split into train/test first — and do it in a way that preserves this ratio in both sets.

# Train-Test-Split

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X=news["content"]
y=news["label"]
X_train, X_test, y_train, y_test=train_test_split(X,y , test_size=0.2, stratify=y, random_state=42)

In [10]:
#Check class imbalance after stratification
y_train.value_counts(normalize=True)*100
y_test.value_counts(normalize=True)*100

label
1    97.39896
0     2.60104
Name: proportion, dtype: float64

## So the class imbalance is preserved after train_test_split

# Handle Class Imbalance

### A few real options, each with different tradeoffs — worth understanding before picking one blindly:

    Class weighting — tell the NB model to weight the minority class more heavily during training. Simplest option, no synthetic data involved. 
    Random oversampling — duplicate minority-class (fake) examples until balanced. Easy, but risks overfitting to those exact duplicated examples.
    SMOTE — synthetically generates new minority examples by interpolating between existing ones in feature space. Standard for tabular data, but there's a subtlety for text: SMOTE was designed for continuous numeric features, and applying it directly to sparse TF-IDF/count vectors is common in practice but statistically shakier than on structured data. Worth knowing this caveat exists even if you still use it.
    Undersampling the majority class — throw away most of the 48K authentic examples to match the ~1K fake count. Simple, but wastes a lot of data — probably not ideal here given how few fake examples you already have.
    Just report the right metrics and skip resampling entirely — for Naive Bayes specifically, since it's a generative model working off priors and likelihoods, an interesting angle is to manually adjust priors rather than resample data at all (NB explicitly separates the prior P(class) from the likelihood, so you can override this cleanly). This ties nicely into "understanding the math" rather than treating resampling as a black-box fix.

    Given this is a from-scratch NB implementation (per our earlier plan), option 5 is actually the most interesting choice for your specific project — you can show you understand why priors matter and manually control them, which most portfolio projects using imblearn.SMOTE() blindly don't demonstrate.

# Base-line Model

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# model = MultinomialNB()
# model.fit(X_train_vec, y_train)

# y_pred = model.predict(X_test_vec)
# print(classification_report(y_test, y_pred))

In [12]:
# Check, if the tokens are meaningfull or not 
vectorizer.get_feature_names_out()[:20]

array(['000', '0054497', '01', '01557229215', '01558136814', '01711',
       '017186959072', '01723236946', '01797170309', '01912990597',
       '01947232734', '01955803728', '0200003244054', '03', '06', '09',
       '0px', '0১', '0২', '10'], dtype=object)

### That confirms it — this is exactly the "garbage tokens" problem I flagged. Your vocabulary right now is dominated by phone numbers, IDs, and stray digits/HTML artifacts (0px is a CSS leftover, probably from scraped web content that wasn't cleaned). None of this is meaningful signal for classifying fake vs authentic news — it's noise that'll bloat your vocabulary and dilute the real word-based patterns NB needs to work with.

    This needs to be fixed before your baseline numbers mean anything. A "97% accuracy" baseline built on phone-number tokens isn't a real baseline — it's noise you'd have to redo anyway.

### Two separate problems mixed together here:
### Numeric noise (phone numbers, IDs, plain digits) — should almost certainly be stripped entirely for this task. News articles don't need digit-strings as classification features (unless you have a specific hypothesis that fake news articles cite phone numbers more often — possible, but treat that as a deliberate feature, not an accident).
### Leftover markup (0px) — signals the raw content text has unstripped HTML/CSS remnants from scraping. This needs proper cleaning, not just tokenization tweaks.

# Clean Text

In [13]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags / leftover markup
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # Remove phone-number-like digit sequences (long digit runs, Bangla + Latin digits)
    text = re.sub(r'[0-9০-৯]{4,}', ' ', text)
    
    # Remove any remaining standalone digits/numbers
    text = re.sub(r'[0-9০-৯]+', ' ', text)
    
    # Remove CSS/artifact fragments like "0px", "10em" etc. (in case any survive)
    text = re.sub(r'\b\w*px\b|\b\w*em\b', ' ', text)
    
    # Keep only Bangla characters (Unicode range) and basic whitespace
    # Bangla Unicode block: U+0980–U+09FF
    text = re.sub(r'[^\u0980-\u09FF\s]', ' ', text)
    
    # Collapse multiple spaces into one, strip leading/trailing
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Check cleaned text

In [14]:
news['content_clean'] = news['content'].apply(clean_text)

# check a few before/after
for i in range(3):
    print("BEFORE:", news['content'].iloc[i][:200])
    print("AFTER :", news['content_clean'].iloc[i][:200])
    print()



BEFORE: গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে (বাকৃবি) উপাচার্যের কার্যালয়ে হট্টগোলের ঘটনায় দুইজনকে সাময়িক বরখাস্ত ও ছয় জনকে শোকজ করেছে বিশ্ববিদ্যালয় প্রশাসন। বুধবার বিশ্ববিদ্যালয় বাকৃবি রেজিস্ট্রার স
AFTER : গত সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে বাকৃবি উপাচার্যের কার্যালয়ে হট্টগোলের ঘটনায় দুইজনকে সাময়িক বরখাস্ত ও ছয় জনকে শোকজ করেছে বিশ্ববিদ্যালয় প্রশাসন বুধবার বিশ্ববিদ্যালয় বাকৃবি রেজিস্ট্রার সাইফুল 

BEFORE: বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রমিক পাঠানোর প্রয়োজনীয় সব ধরনের ব্যবস্থা নেয়ার সুপারিশ করেছে সংসদীয় কমিটি। ২৫ সেপ্টেম্বর প্রবাসী কল্যাণ ও বৈদেশিক কর্মসংস্থান মন্ত্রণালয়ের মন্ত্রীর নেতৃত্বে
AFTER : বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রমিক পাঠানোর প্রয়োজনীয় সব ধরনের ব্যবস্থা নেয়ার সুপারিশ করেছে সংসদীয় কমিটি সেপ্টেম্বর প্রবাসী কল্যাণ ও বৈদেশিক কর্মসংস্থান মন্ত্রণালয়ের মন্ত্রীর নেতৃত্বে মাল

BEFORE: নরসিংদীর মনোহরদীতে প্রেমের প্রস্তাবে রাজি না হওয়ায় তানজিমা (১৪) নামে এক স্কুলছাত্রীকে কুপিয়ে জখম করেছে মফিজুল ইসলাম (১৮) নামে এক যুবক। বুধবার সকালে উপজেলা

In [15]:
# check for empty strings after cleaning (bad sign if many)
print((news['content_clean'].str.strip() == '').sum())

6


# Only 6 rows become empty. 6 out of 49k is negligble, we can remove these

In [16]:
news = news[news['content_clean'].str.strip() != ''].reset_index(drop=True)

# Retrain Model

In [17]:
X = news['content_clean']
y = news['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# confirm balance held
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

label
1    97.400941
0     2.599059
Name: proportion, dtype: float64
label
1    97.398699
0     2.601301
Name: proportion, dtype: float64


# Vectorize Again

In [18]:
vectorizer = CountVectorizer(token_pattern=r'[\u0980-\u09FF]+', min_df=5)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(vectorizer.get_feature_names_out()[:20])

['অ' 'অঁতনি' 'অঁরি' 'অং' 'অংক' 'অংকটা' 'অংকন' 'অংকে' 'অংকের' 'অংগ'
 'অংগীকার' 'অংশ' 'অংশই' 'অংশও' 'অংশকে' 'অংশগুলো' 'অংশগ্রহণ' 'অংশগ্রহণকারী'
 'অংশগ্রহণকারীদের' 'অংশগ্রহণকারীর']


In [19]:
# Check frequency of these suspicious tokens
word_counts = X_train_vec.sum(axis=0)
word_freq = dict(zip(vectorizer.get_feature_names_out(), word_counts.tolist()[0]))

# sort by frequency, look at the bottom (rare) and top (common)
sorted_words = sorted(word_freq.items(), key=lambda x: -x[1])
print("Top 20 most frequent:", sorted_words[:20])
print("Bottom 20 least frequent:", sorted_words[-20:])

Top 20 most frequent: [('ও', 122422), ('করে', 93593), ('এ', 75434), ('না', 72140), ('থেকে', 69673), ('করা', 61578), ('বলেন', 60066), ('এই', 58900), ('তিনি', 57003), ('হয়', 56015), ('হয়েছে', 51139), ('হবে', 49921), ('এবং', 44727), ('তার', 42637), ('জন্য', 40099), ('নিয়ে', 39013), ('করতে', 37089), ('করেন', 36729), ('এক', 34495), ('একটি', 33754)]
Bottom 20 least frequent: [('হোসেনই', 5), ('হোয়াই', 5), ('হোয়েন', 5), ('হ্মন্দী', 5), ('হ্যাংওভার', 5), ('হ্যাকিংয়ের', 5), ('হ্যাটস', 5), ('হ্যান্ডগান', 5), ('হ্যান্ডশেক', 5), ('হ্যামস্ট্রিং', 5), ('হ্যামস্ট্রিংয়ে', 5), ('হ্যারল্ড', 5), ('হ্যারিক্যানের', 5), ('হয়েছেই', 5), ('্যাঙ্ক', 5), ('্যাঙ্কিংয়েও', 5), ('্যাচেল', 5), ('ড়ের', 5), ('য়ু', 5), ('য়ো', 5)]


# Fit Base Model

In [20]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print("Vocab size:", len(vectorizer.get_feature_names_out()))

              precision    recall  f1-score   support

           0       0.27      0.86      0.42       260
           1       1.00      0.94      0.97      9735

    accuracy                           0.94      9995
   macro avg       0.64      0.90      0.69      9995
weighted avg       0.98      0.94      0.95      9995

Vocab size: 59897


### This is your real baseline now — and it's actually a genuinely interesting result to unpack, not just "run and move on."
### What's happening here:
    Recall on fake (class 0) = 0.86 — surprisingly high already. NB is catching 86% of actual fake news articles.
    Precision on fake (class 0) = 0.27 — but only 27% of what it flags as fake actually is fake. So it's over-triggering — lots of false alarms, real authentic articles getting mislabeled as fake.
    Overall accuracy = 0.94 — looks strong, but you already know from our imbalance discussion not to trust this number alone. A trivial "always predict authentic" model would hit 97.4%, so 94% is actually slightly worse than doing nothing, in raw accuracy terms — but it's making a real tradeoff (catching fakes) rather than being a useless classifier.

### Why did this happen even with the imbalanced empirical prior baked in (2.6% prior for fake)? This is worth sitting with for a second, since it's a bit counter to naive intuition. The likelihood term is doing a lot of the work here — fake news articles apparently have distinctive enough word patterns (sensational language, particular phrasing) that the likelihood ratio overpowers the weak prior for many documents. NB isn't just "voting with the prior" — the word evidence is strong enough to flip many decisions toward fake despite the prior working against it.

# Prior Adjustment

In [21]:
model.class_log_prior_

array([-3.65002056, -0.02633432])

In [22]:
new_priors = np.array([np.log(0.80), np.log(0.20)])
model.class_log_prior_ = new_priors

y_pred_adjusted = model.predict(X_test_vec)
print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.25      0.87      0.39       260
           1       1.00      0.93      0.96      9735

    accuracy                           0.93      9995
   macro avg       0.62      0.90      0.68      9995
weighted avg       0.98      0.93      0.95      9995



In [23]:
# vectorize headlines instead of content
X_train_headline, X_test_headline, y_train_h, y_test_h = train_test_split(
    news['headline'], y, test_size=0.2, stratify=y, random_state=42
)

vectorizer_h = CountVectorizer(token_pattern=r'[\u0980-\u09FF]+', min_df=5)
X_train_h_vec = vectorizer_h.fit_transform(X_train_headline)
X_test_h_vec = vectorizer_h.transform(X_test_headline)

model_h = MultinomialNB()
model_h.fit(X_train_h_vec, y_train_h)

y_pred_h = model_h.predict(X_test_h_vec)
print("Baseline (headline):")
print(classification_report(y_test_h, y_pred_h))

Baseline (headline):
              precision    recall  f1-score   support

           0       0.54      0.52      0.53       260
           1       0.99      0.99      0.99      9735

    accuracy                           0.98      9995
   macro avg       0.76      0.75      0.76      9995
weighted avg       0.98      0.98      0.98      9995



In [24]:
print(vectorizer_h.get_feature_names_out()[:20])

['অংশ' 'অংশগ্রহণ' 'অংশগ্রহণমূলক' 'অংশগ্রহণে' 'অংশগ্রহণের' 'অক্টোবর'
 'অক্টোবরে' 'অক্টোবরের' 'অক্ষত' 'অক্ষয়' 'অখুশি' 'অগাস্ট' 'অগ্নিকাণ্ড'
 'অগ্নিকাণ্ডে' 'অগ্নিসংযোগ' 'অগ্ন্যুৎপাত' 'অগ্রগতি' 'অগ্রণী' 'অঙ্কুশ'
 'অঙ্গনে']


# Prior Adjustment for small text(Headline)

In [25]:
print(model_h.class_log_prior_)

[-3.65002056 -0.02633432]


In [26]:
new_priors_h = np.array([np.log(0.5), np.log(0.5)])
model_h.class_log_prior_ = new_priors_h

y_pred_h_adjusted = model_h.predict(X_test_h_vec)
print(classification_report(y_test_h, y_pred_h_adjusted))

              precision    recall  f1-score   support

           0       0.11      0.82      0.19       260
           1       0.99      0.82      0.90      9735

    accuracy                           0.82      9995
   macro avg       0.55      0.82      0.54      9995
weighted avg       0.97      0.82      0.88      9995



# Prior adjustment Conclusion

### Prior adjustment in Naive Bayes is not a universally useful lever — its effectiveness is inversely related to document length. On full-article content (hundreds of words), the accumulated log-likelihood evidence overwhelms even a 50/50 prior shift. On headlines (10 words), the same shift causes recall to jump 30 points at the cost of precision. This has practical implications: if you want a tunable fake-news filter, prior-tuning should be applied on short-text models, not long-form content models.

# Let's try Resampling

### Headlines are optimized for clicks/attention (by both legitimate and fake sources), so they're a noisier signal for genuine authenticity, whereas full content is where fabricated claims, sourcing patterns, and rhetorical style actually show up. That's why we are moving forward with resampling.

In [27]:
# Try random oversampling first, since it's the simplest to reason about and implement:
from sklearn.utils import resample
train_news= pd.DataFrame({'text': X_train, 'label': y_train})
majority= train_news[train_news.label==1]
minority= train_news[train_news.label==0]
minority_upsample=resample(minority, replace=True, n_samples=len(majority), random_state=42)
news_balanced= pd.concat([majority, minority_upsample])

In [28]:
news_balanced.shape

(77874, 2)

In [29]:
news_balanced['label'].value_counts(normalize=True)*100

label
1    50.0
0    50.0
Name: proportion, dtype: float64

# Re-build Model on Balanced data

In [30]:
vectorizer_bal = CountVectorizer(token_pattern=r'[\u0980-\u09FF]+', min_df=5)
X_train_bal_vec = vectorizer_bal.fit_transform(news_balanced['text'])
y_train_bal = news_balanced['label']

model_bal = MultinomialNB()
model_bal.fit(X_train_bal_vec, y_train_bal)

# IMPORTANT: transform test set with vectorizer_bal, not the old vectorizer
X_test_vec_bal = vectorizer_bal.transform(X_test)

y_pred_bal = model_bal.predict(X_test_vec_bal)
print(classification_report(y_test, y_pred_bal))

              precision    recall  f1-score   support

           0       0.47      0.82      0.60       260
           1       1.00      0.98      0.99      9735

    accuracy                           0.97      9995
   macro avg       0.73      0.90      0.79      9995
weighted avg       0.98      0.97      0.98      9995



In [31]:
model_bal.predict(X_test_vec_bal[10])

array([1])

In [32]:
X_test[10]

'বছর বয়সী রোকসানার জীবন দুর্বিসহ করে তুলেছে রাজধানীর ওয়ারী এলাকার পাষণ্ড এক দম্পতি ধর্ষণ শারীরিক ও মানসিক নির্যাতন না খেয়ে রাখাসহ এমন কোনো নির্যাতন বাদ ছিল না যা করা হয়নি তার ওপর গত আট মাস ধরে প্রতিনিয়ত নির্যাতনের কারণে শরীরের বিভিন্ন অংশের চামড়া পচে গেছে তার সেইসঙ্গে শরীরের চামড়া তার হাড়গুলোকে লেপ্টে ধরেছে প্রথম দেখাতেই যে কেউ ভেবে বসবে এটি একটি জীবন্ত কঙ্কাল এছাড়াও সারা শরীরে ক্ষতস্থানের শেষ নেই সেই দাগগুলোই স্পস্ট করে দিয়েছে কী কী কৌশলে তার ওপর নির্যাতন চালানো হয়েছে রোকসানা বর্তমানে ঢাকা মেডিকেল কলেজ হাসপাতালের বার্ন অ্যান্ড প্লাস্টিক ইউনিটের য় তলার আইসিইউ শিশু ওয়ার্ডের নম্বর বেডে চিকিৎসাধীন গত দিন ধরে রোকসানার চিকিৎসা ও তাকে নির্যাতনকারীদের বিচারের দাবিতে সামাজিক যোগাযোগ মাধ্যমে ফেসবুকে সোচ্চার ভূমিকা পালন করছেন ঢাকায় বসবাসকারী নড়াইলের বাসিন্দা আমিরুল ইসলাম লিটু বিষয়টি ফেসবুকে ভাইরাল হয়ে যাওয়ায় অনেকেই রোকসানার পাশে এসে দাঁড়িয়েছেন ইতোমধ্যে তাকে আইনি ও চিকিৎসা সহায়তাও দেয়া শুরু হয়েছে প্রতিবারের ন্যায় এবারও অসহায় রোকসানাকে দেখতে মঙ্গলবার সন্ধ্যায় ঢাকা মেডিকেলে হাজির হন বাংলাদেশ ছাত্রল

# Test on unseen data (bdnews24.com)

In [52]:
new_articles = ["ফেরিতে অ্যাম্বুল্যান্সের রোগীসহ আড়াইশ’ যাত্রীর উদ্বেগের রাত।চট্টগ্রাম: সন্দ্বীপ চ্যানেলে ডুবোচরে আটকে পড়েছিল ‘ফেরি কপোতাক্ষ’। অ্যাম্বুল্যান্সে থাকা রোগীসহ প্রায় আড়াইশ’ যাত্রী ছিলেন ওই ফেরিতে।ফেরি থেকে নেমে ঘরে ফেরার উপায় ছিল না। শেষ ভরসা জোয়ারের অপেক্ষায় কেটেছে তাদের উদ্বেগের রাত।ক্যানটিনে ছিল না খাবার। রাতভর যাত্রীদের দুশ্চিন্তা,দুর্ভোগ এবং স্বজন ও দায়িত্বশীলদের নির্ঘুম রাত কেটেছে।বুধবার (২ সেপ্টেম্বর) সন্ধ্যা ৬টার দিকে সীতাকুণ্ডের বাঁশবাড়িয়া যাওয়ার পথে ডুবোচরে আটকে পড়া ফেরিটি রাত ২টা ৪৫ মিনিটে পুনরায় যাত্রা শুরু করে। রাত সোয়া ৪টার দিকে বাঁশবাড়িয়া ফেরিঘাটে পৌঁছালেও অতিরিক্ত জোয়ারের কারণে বাস ও ব্যক্তিগত গাড়ি ফেরি থেকে নামাতে অপেক্ষা করতে হয় কয়েক ঘণ্টা।ফেরিতে প্রয়োজনীয় খাদ্য ও খাবার পানির সংকটের খবর ছড়িয়ে পড়লে রাত ৯টার পর যাত্রীদের জন্য শুকনো খাবার পৌঁছে দেওয়ার উদ্যোগ নেওয়া হয়। গভীর রাতে চট্টগ্রাম-৩ (সন্দ্বীপ) আসনের সংসদ সদস্য মোস্তফা কামাল পাশার পৃষ্ঠপোষকতায় বিএনপির নেতাকর্মীরা রান্না করা খাবার ও পানি ফেরিতে নিয়ে যান।এর আগে সন্দ্বীপ উপজেলা নির্বাহী কর্মকর্তা আমজাদ হোসেন ফেরিতে গিয়ে আটকে থাকা যাত্রীদের খোঁজখবর নেন।সন্দ্বীপের একটি সরকারি প্রাথমিক বিদ্যালয়ের প্রধান শিক্ষক দিদারুল আলম বলেন, ‘আমার মেয়েকে ফেরিতে পাঠিয়ে সারা রাত উৎকণ্ঠার মধ্যে কাটিয়েছি।’বাংলাদেশ অভ্যন্তরীণ নৌপরিবহন কর্তৃপক্ষের উপ-পরিচালক (ভারপ্রাপ্ত) নয়ন শীল বাংলানিউজকে বলেন, রোগীসহ অ্যাম্বুল্যান্স আসাতে ফেরি মাস্টার যাত্রা শুরু করেছিলেন। সন্দ্বীপের সংসদ সদস্য ফেরির যাত্রীদের রাতে খাবার ও পানির ব্যবস্থা করেছেন। প্রশাসন ও আইনশৃঙ্খলা বাহিনী যাত্রীদের তৎপরতায় নিয়োজিত ছিলেন।"]

new_articles_clean = [clean_text(a) for a in new_articles]
new_vec = vectorizer_bal.transform(new_articles_clean)  
predictions = model_bal.predict(new_vec)
probs = model_bal.predict_proba(new_vec)

for text, pred, prob in zip(new_articles, predictions, probs):
    print(f"Predicted: {'Fake' if pred == 0 else 'Authentic'} | Confidence: {prob}")    
    print(text[:1000])
    print()

Predicted: Authentic | Confidence: [3.20807333e-115 1.00000000e+000]
ফেরিতে অ্যাম্বুল্যান্সের রোগীসহ আড়াইশ’ যাত্রীর উদ্বেগের রাত।চট্টগ্রাম: সন্দ্বীপ চ্যানেলে ডুবোচরে আটকে পড়েছিল ‘ফেরি কপোতাক্ষ’। অ্যাম্বুল্যান্সে থাকা রোগীসহ প্রায় আড়াইশ’ যাত্রী ছিলেন ওই ফেরিতে।ফেরি থেকে নেমে ঘরে ফেরার উপায় ছিল না। শেষ ভরসা জোয়ারের অপেক্ষায় কেটেছে তাদের উদ্বেগের রাত।ক্যানটিনে ছিল না খাবার। রাতভর যাত্রীদের দুশ্চিন্তা,দুর্ভোগ এবং স্বজন ও দায়িত্বশীলদের নির্ঘুম রাত কেটেছে।বুধবার (২ সেপ্টেম্বর) সন্ধ্যা ৬টার দিকে সীতাকুণ্ডের বাঁশবাড়িয়া যাওয়ার পথে ডুবোচরে আটকে পড়া ফেরিটি রাত ২টা ৪৫ মিনিটে পুনরায় যাত্রা শুরু করে। রাত সোয়া ৪টার দিকে বাঁশবাড়িয়া ফেরিঘাটে পৌঁছালেও অতিরিক্ত জোয়ারের কারণে বাস ও ব্যক্তিগত গাড়ি ফেরি থেকে নামাতে অপেক্ষা করতে হয় কয়েক ঘণ্টা।ফেরিতে প্রয়োজনীয় খাদ্য ও খাবার পানির সংকটের খবর ছড়িয়ে পড়লে রাত ৯টার পর যাত্রীদের জন্য শুকনো খাবার পৌঁছে দেওয়ার উদ্যোগ নেওয়া হয়। গভীর রাতে চট্টগ্রাম-৩ (সন্দ্বীপ) আসনের সংসদ সদস্য মোস্তফা কামাল পাশার পৃষ্ঠপোষকতায় বিএনপির নেতাকর্মীরা রান্না করা খাবার ও পানি ফেরিতে নিয়ে যা

# Test on unseen data (earki.com)

In [50]:
new_articles=["আমি শাবানা। আমার স্বামী জসিম। খোকন নামে আমাদের একটি ছেলে আছে। খোকন স্কুলে পড়ে।জসিম ভ্যানে করে আচার বিক্রি করে অর্থোপার্জন করে। আমাদের সংসারে সুখের অভাব নেই। কিন্তু আমাদের আর্থিক অবস্থা ভালো না- চ্যাটিং করতেই এমবি ফুরোয়।এই অল্প উপার্জন থেকেই জসিম মাসে মাসে গ্রামের বাড়িতে টাকা পাঠায়। আমার শ্বশুর আনোয়ার হোসেন অসুস্থ, হার্টের রোগী। দেবর কাঞ্চন কলেজে বিএ পড়ে। তার ওপর জসিম লটারি কিনে টাকা নষ্ট করে।একদিন পাশের বাসার এক ভাবি এসে আমাকে ফেসবুকে একটি গ্রুপে এড করে দেন। তিনি দেখিয়ে দেন কীভাবে উদ্দোক্তা হতে হয়। তারপর আমি গ্রুপের স্যারের পরামর্শ নিই।জসিম রাতে বাড়ি ফিরলে আমি তাকে বুঝিয়ে বলি কীভাবে বড়লোক হওয়া যায়। তারপর ভাবিদের গ্রুপটার কথা বলি। জসিম বিষয়টি বুঝতে পারে। আমি জসিমকেও সেই গ্রুপে এড করে দেই।তারপর জসিম স্যারের পরামর্শে আমাকে একটি সেলাইমেশিন কিনে দেয়। আমি রাতদিন কাপড় সেলাই করি। সেখান থেকে টাকা জমিয়ে জসিমকে একটা টেম্পো কিনে দেই।ধীরেধীরে আমি হাতে বানানো জামা অনলাইনে বিক্রি করতে থাকি। জসিম হোম-ডেলিভারি দেয়। জসিমও ভালো রোজগার করতে থাকে। এভাবেই আমাদের অনেক টাকা জমে যায়। তারপর আমি একদিন দশটি সেলাইমেশিন কিনে দশজন নারীকে নিয়ে ছোট্ট একটা প্রতিষ্ঠান খুলি।জসিম তার টেম্পো বিক্রি করে একটি টয়োটা করোলা কিনে ফেলে! আমার প্রতিষ্ঠানও বড় হতে থাকে। অবশেষে আমি আর জসিম মিলে একটি গার্মেন্টস ফ্যাক্টরি চালু করলাম।সেদিন যদি ভাবি আমাকে ঐ গ্রুপে এড না করতেন তবে আমি শাবানা আজ এতো দূর আসতে পারতাম না। স্যার আর প্রিয় গ্রুপকে অনেক ধন্যবাদ। স্যারের গ্রুপের কারণেই আমি শাবানা আজ কোটিপতি হতে পেরেছি।"]
new_articles_clean = [clean_text(a) for a in new_articles]
new_vec = vectorizer_bal.transform(new_articles_clean)  
predictions = model_bal.predict(new_vec)
probs = model_bal.predict_proba(new_vec)

for text, pred, prob in zip(new_articles, predictions, probs):
    print(f"Predicted: {'Fake' if pred == 0 else 'Authentic'} | Confidence: {prob}")
    print(text[:1000])
    print()

Predicted: Fake | Confidence: [1.00000000e+00 2.20401501e-15]
আমি শাবানা। আমার স্বামী জসিম। খোকন নামে আমাদের একটি ছেলে আছে। খোকন স্কুলে পড়ে।জসিম ভ্যানে করে আচার বিক্রি করে অর্থোপার্জন করে। আমাদের সংসারে সুখের অভাব নেই। কিন্তু আমাদের আর্থিক অবস্থা ভালো না- চ্যাটিং করতেই এমবি ফুরোয়।এই অল্প উপার্জন থেকেই জসিম মাসে মাসে গ্রামের বাড়িতে টাকা পাঠায়। আমার শ্বশুর আনোয়ার হোসেন অসুস্থ, হার্টের রোগী। দেবর কাঞ্চন কলেজে বিএ পড়ে। তার ওপর জসিম লটারি কিনে টাকা নষ্ট করে।একদিন পাশের বাসার এক ভাবি এসে আমাকে ফেসবুকে একটি গ্রুপে এড করে দেন। তিনি দেখিয়ে দেন কীভাবে উদ্দোক্তা হতে হয়। তারপর আমি গ্রুপের স্যারের পরামর্শ নিই।জসিম রাতে বাড়ি ফিরলে আমি তাকে বুঝিয়ে বলি কীভাবে বড়লোক হওয়া যায়। তারপর ভাবিদের গ্রুপটার কথা বলি। জসিম বিষয়টি বুঝতে পারে। আমি জসিমকেও সেই গ্রুপে এড করে দেই।তারপর জসিম স্যারের পরামর্শে আমাকে একটি সেলাইমেশিন কিনে দেয়। আমি রাতদিন কাপড় সেলাই করি। সেখান থেকে টাকা জমিয়ে জসিমকে একটা টেম্পো কিনে দেই।ধীরেধীরে আমি হাতে বানানো জামা অনলাইনে বিক্রি করতে থাকি। জসিম হোম-ডেলিভারি দেয়। জসিমও ভালো রোজগার করতে থাক